### Feature Engineering
The following features will be engineered to enhance the recommendation model:

1. **Review_Depth**: Word count of the review text. Longer reviews often provide more detailed information.
2. **Readability_Score**: Computed using Flesch Reading Ease; indicates how accessible the text is.
3. **Rating_Extremity**: Measures how far a rating deviates from the median (3), calculated as $|Rating - 3|$.
4. **Sentiment_Polarity**: Sentiment score (Compound) derived from the VADER analyzer.
5. **Review_Age_Days**: Number of days since the review was posted relative to the most recent review in the dataset.
6. **Wilson_Helpfulness**: Adjusted helpfulness score using the Wilson Lower Bound to ensure statistical fairness.

### Step 1: Load Dataset
Initialize the environment and load the Amazon review data.

In [1]:
import pandas as pd

df = pd.read_csv("Reviews_withURL.csv")
print(df.columns)
print(df.head())

Index(['Unnamed: 0', 'Id', 'ProductId', 'UserId', 'ProfileName',
       'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time',
       'Summary', 'Text', 'ProductURL'],
      dtype='str')
   Unnamed: 0  Id   ProductId          UserId  \
0           0   1  B001E4KFG0  A3SGXH7AUHU8GW   
1           1   2  B00813GRG4  A1D87F6ZCVE5NK   
2           2   3  B000LQOCH0   ABXLMWJIXXAIN   
3           3   4  B000UA0QIQ  A395BORC6FGVXV   
4           4   5  B006K2ZZ7K  A1UQRSCLF8GW1T   

                       ProfileName  HelpfulnessNumerator  \
0                       delmartian                     1   
1                           dll pa                     0   
2  Natalia Corres "Natalia Corres"                     1   
3                             Karl                     3   
4    Michael D. Bigham "M. Wassir"                     0   

   HelpfulnessDenominator  Score        Time                Summary  \
0                       1      5  1303862400  Good Quality Dog Food   
1  

### Step 2: Data Preprocessing
Perform basic cleaning. This step is crucial for **data validation and error prevention**, ensuring numerical columns are properly formatted and null values are handled.

In [2]:
df["Text"] = df["Text"].fillna("").astype(str)
df["Score"] = pd.to_numeric(df["Score"], errors="coerce")
df["HelpfulnessNumerator"] = pd.to_numeric(df["HelpfulnessNumerator"], errors="coerce").fillna(0)
df["HelpfulnessDenominator"] = pd.to_numeric(df["HelpfulnessDenominator"], errors="coerce").fillna(0)
df["Time"] = pd.to_numeric(df["Time"], errors="coerce")

### Step 3: Compute Review Depth
The assumption is that longer reviews typically contain more informative insights.

In [3]:
df["Review_Depth"] = df["Text"].apply(lambda x: len(str(x).split()))

### Step 4: Compute Readability Score
- **High Score**: Easy to read.
- **Low Score**: Complex sentences, harder to understand.

In [4]:
!pip install textstat -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import textstat

df["Readability_Score"] = df["Text"].apply(
    lambda x: textstat.flesch_reading_ease(str(x))
)

### Step 5: Compute Rating Extremity
Calculated as $E = |Rating - 3|$ to determine how polarized a review is compared to the neutral baseline.

In [6]:
df["Rating_Extremity"] = df["Score"].apply(lambda x: abs(x - 3))

### Step 6: Sentiment Analysis
Using NLTK's VADER to determine the emotional polarity of the review text.

In [7]:
pip install nltk -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import nltk
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [9]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df["Sentiment_Polarity"] = df["Text"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

### Step 7: Review Age (Days)
Older reviews tend to accumulate more "helpful" votes. Calculating the age allows us to control for this temporal bias.

In [10]:
df["Review_Date"] = pd.to_datetime(df["Time"], unit="s", errors="coerce")
latest_date = df["Review_Date"].max()

df["Review_Age_Days"] = (latest_date - df["Review_Date"]).dt.days

### Step 8: Wilson Score Implementation
We avoid using raw helpfulness ratios (Helpful / Total) because they are biased toward reviews with very few votes (e.g., 1/1 vs. 90/100). The Wilson Lower Bound provides a statistically more reliable helpfulness metric.

### Correlation Analysis
Analyze which engineered features correlate most strongly with the **Wilson Helpfulness** score. This helps answer:
* Does word count (Depth) correlate with helpfulness?
* Do extreme ratings receive fewer helpful votes?
* Does readability impact user engagement?

In [11]:
import numpy as np

def wilson_lower_bound(helpful, total, z=1.96):
    helpful = float(helpful)
    total = float(total)

    if total == 0:
        return 0.0

    p_hat = helpful / total

    denominator = 1 + z**2 / total
    centre = p_hat + z**2 / (2 * total)
    margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * total)) / total)

    return (centre - margin) / denominator

In [12]:
df["Wilson_Helpfulness"] = df.apply(
    lambda row: wilson_lower_bound(
        row["HelpfulnessNumerator"],
        row["HelpfulnessDenominator"]
    ),
    axis=1
)

C:\Users\User\AppData\Local\Temp\ipykernel_27000\2121843692.py:14: RuntimeWarning: invalid value encountered in sqrt
  margin = z * np.sqrt((p_hat * (1 - p_hat) + z**2 / (4 * total)) / total)


### Step 9: Final Product Weight Calculation
The final product weight is calculated using a weighted average of the Wilson Score (weighted by Review Depth) and scaled by the logarithm of the total review count.

### Aggregating Data by Product ID
Grouping individual reviews to compute the final weights for each unique product.

### Final Weight Export
Reviewing the descriptive statistics of the generated `product_weight` before exporting to a CSV file.

In [13]:
cols = [
    "Review_Depth",
    "Readability_Score",
    "Rating_Extremity",
    "Sentiment_Polarity",
    "Review_Age_Days",
    "Wilson_Helpfulness"
]

print(df[cols].corr()["Wilson_Helpfulness"].sort_values(ascending=False))

Wilson_Helpfulness    1.000000
Review_Age_Days       0.336415
Review_Depth          0.188840
Rating_Extremity      0.053048
Sentiment_Polarity    0.014126
Readability_Score    -0.088501
Name: Wilson_Helpfulness, dtype: float64


product_weight = (
    Σ (Wilson_Helpfulness × Review_Depth)
    / Σ (Review_Depth)
) * [log(1 + review_count)]

In [14]:
print(df[["ProductId", "Wilson_Helpfulness", "Review_Depth"]].head())

    ProductId  Wilson_Helpfulness  Review_Depth
0  B001E4KFG0            0.206543            48
1  B00813GRG4            0.000000            31
2  B000LQOCH0            0.206543            94
3  B000UA0QIQ            0.438494            41
4  B006K2ZZ7K            0.000000            27


In [15]:
product_weight_df = (
    df.groupby("ProductId")
      .apply(lambda x: pd.Series({
          "weighted_wilson": (
              (x["Wilson_Helpfulness"] * x["Review_Depth"]).sum()
              / x["Review_Depth"].sum()
          ),
          "review_count": len(x)
      }))
      .reset_index()
)

In [16]:
import numpy as np

epsilon = 0.01

product_weight_df["product_weight"] = (
    (product_weight_df["weighted_wilson"] + epsilon) *
    np.log1p(product_weight_df["review_count"])
)

In [17]:
print("origin df number of rows:", len(df))
print("product_weight_df number of rows:", len(product_weight_df))

origin df number of rows: 568454
product_weight_df number of rows: 74258


In [18]:
final_product_weight = product_weight_df[[
    "ProductId",
    "product_weight"
]]

In [19]:
print(product_weight_df["product_weight"].describe())

count    74258.000000
mean         0.245532
std          0.298971
min          0.006931
25%          0.006931
50%          0.150096
75%          0.367985
max          3.221922
Name: product_weight, dtype: float64


In [20]:
final_product_weight.to_csv("product_weight.csv", index=False)